# ML Project / NASDAQ Prediction and Analysis

**Objective** : Predicting possible long/short entries on the NASDAQ index, using Machine Learning algorithms and other tools to carry this project.


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import requests 
import io

In [11]:
html = requests.get("https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies", headers={"User-Agent": "Mozilla/5.0"}).text
table = pd.read_html(io.StringIO(html))
ndx = next(t for t in table if "Ticker" in t.columns)
TICKERS = ndx["Ticker"].str.replace(".", "-", regex=False).tolist()

In [27]:
len(TICKERS)

101

In [31]:
test = yf.download("AAPL", start="2018-01-01", end="2025-12-31",
                     auto_adjust=True, progress=False)
print(test.columns)
print(test.tail())

MultiIndex([( 'Close', 'AAPL'),
            (  'High', 'AAPL'),
            (   'Low', 'AAPL'),
            (  'Open', 'AAPL'),
            ('Volume', 'AAPL')],
           names=['Price', 'Ticker'])
Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2025-12-23  271.620667  271.760301  268.828279  270.104803  29642000
2025-12-24  273.066742  274.682340  271.461127  271.600731  17910600
2025-12-26  272.657806  274.622460  272.119264  273.415753  21521800
2025-12-29  273.016876  273.615223  271.610700  271.949773  23715200
2025-12-30  272.338684  273.335969  271.540868  272.069428  22139600


In [16]:
raw = yf.download(TICKERS, start="2018-01-01", end="2025-12-31",
                  auto_adjust=True, group_by="ticker",
                  progress=False, threads=True)

$HONA: Data doesn't exist for startDate = 1514782800, endDate = 1767157200
$SPCX: Data doesn't exist for startDate = 1514782800, endDate = 1767157200

2 Failed downloads:
['HONA', 'SPCX']: Data doesn't exist for startDate = 1514782800, endDate = 1767157200


In [23]:
raw['SPCX'].head()

Price,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
coverage_rows = []
available = raw.columns.get_level_values(0)
for ticker in TICKERS:
    if ticker in available:
        d = raw[ticker].dropna(how="all")
        coverage_rows.append({"ticker": ticker, "days": len(d),
                      "first_date": d.index.min().date() if len(d) else None,
                      "last_date": d.index.max().date() if len(d) else None})
    else:
        coverage_rows.append({"ticker": ticker, "days": 0, "first_date": None, "last_date": None})

coverage = pd.DataFrame(coverage_rows)

print(coverage.sort_values("days").head(10))   
print("Tickers without data:", (coverage["days"] == 0).sum())

   ticker  days  first_date   last_date
43   HONA     0        None        None
87   SPCX     0        None        None
29   CRWV   191  2025-03-28  2025-12-30
84   SNDK   221  2025-02-13  2025-12-30
66   NBIS   299  2024-10-21  2025-12-30
15   ALAB   447  2024-03-20  2025-12-30
13    ARM   576  2023-09-14  2025-12-30
41   GEHC   762  2022-12-15  2025-12-30
27    CEG   991  2022-01-19  2025-12-30
12    APP  1184  2021-04-15  2025-12-30
Tickers without data: 2


In [33]:
print(coverage.sort_values("days").head(20))

   ticker  days  first_date   last_date
43   HONA     0        None        None
87   SPCX     0        None        None
29   CRWV   191  2025-03-28  2025-12-30
84   SNDK   221  2025-02-13  2025-12-30
66   NBIS   299  2024-10-21  2025-12-30
15   ALAB   447  2024-03-20  2025-12-30
13    ARM   576  2023-09-14  2025-12-30
41   GEHC   762  2022-12-15  2025-12-30
27    CEG   991  2022-01-19  2025-12-30
12    APP  1184  2021-04-15  2025-12-30
2    ABNB  1269  2020-12-10  2025-12-30
36   DASH  1270  2020-12-09  2025-12-30
81   RKLB  1280  2020-11-24  2025-12-30
73   PLTR  1319  2020-09-30  2025-12-30
33   DDOG  1579  2019-09-19  2025-12-30
31   CRWD  1648  2019-06-12  2025-12-30
62   MRNA  1775  2018-12-07  2025-12-30
77    PDD  1868  2018-07-26  2025-12-30
14   ASML  2010  2018-01-02  2025-12-30
9     ADI  2010  2018-01-02  2025-12-30


In [35]:
df = yf.download('^NDX')
df = pd.DataFrame(df)

[*********************100%***********************]  1 of 1 completed
